# Phase 3 — LegalIR strong rerankers on RTX Pro 6000 (offline)
Attach a competition-data dataset containing `train.json`, `private-official.json`, and `selected-contexts/selected-contexts/`, plus the Phase 2 Harrier bundle and Phase 3 reranker delta bundle. Set Internet Off. By default this runs private inference only; private Recall cannot be computed without labels. Its test-specific working directory is preserved when rerunning cells in the same live session so cached pipeline work can resume.

In [ ]:
import os
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
from pathlib import Path
EXPERIMENT_ID = 'phase3-rerankers-harrier-retrieval'
DATASET_DIR = Path('/kaggle/input/REPLACE_WITH_COMPETITION_DATASET_SLUG')
TEST_FILENAME = 'private-official.json'  # switch to public-official.json for public inference
if TEST_FILENAME not in {'private-official.json', 'public-official.json'}: raise ValueError(f'Unsupported test input: {TEST_FILENAME}')
TEST_LABEL = 'private' if TEST_FILENAME == 'private-official.json' else 'public'
CHECKPOINT_ARTIFACTS_DIR = None  # Phase 3 checkpoint, or a Phase 2 checkpoint to reuse retrieval indexes
PHASE2_BUNDLE = Path('/kaggle/input/REPLACE_WITH_PHASE2_HARRIER_BUNDLE_SLUG/legalir-phase2-harrier-bundle')
DELTA_BUNDLE = Path('/kaggle/input/REPLACE_WITH_PHASE3_RERANKER_DELTA_SLUG/legalir-phase3-reranker-delta')
WORK_DIR = Path(f'/kaggle/working/legalir-phase3-{TEST_LABEL}-run')
RUNTIME_DIR = Path('/kaggle/working/legalir-phase3-runtime')

In [ ]:
import json, shutil, subprocess, sys, time
def run(*command, cwd=None, env=None):
    print('+', ' '.join(map(str, command)))
    started = time.perf_counter()
    subprocess.run(list(map(str, command)), cwd=cwd, env=env, check=True)
    print(f'Completed in {(time.perf_counter() - started) / 60:.1f} minutes')
delta_manifest = json.loads((DELTA_BUNDLE / 'manifests' / 'bundle_manifest.json').read_text(encoding='utf-8'))
if delta_manifest.get('experiment_id') != EXPERIMENT_ID: raise RuntimeError(f"Wrong Phase 3 bundle: {delta_manifest.get('experiment_id')}")
phase2_manifest_path = PHASE2_BUNDLE / 'manifests' / 'bundle_manifest.json'
if not phase2_manifest_path.is_file(): raise FileNotFoundError(f'Missing Phase 2 manifest: {phase2_manifest_path}')
phase2_manifest = json.loads(phase2_manifest_path.read_text(encoding='utf-8'))
phase2_names = {row['name'] for row in phase2_manifest['models']}
if not {'vietlegal_harrier', 'vietnamese_embedding', 'nemotron'} <= phase2_names: raise RuntimeError(f'Phase 2 bundle lacks retrievers: {phase2_names}')
for record in delta_manifest['files']:
    path = DELTA_BUNDLE / record['path']
    if not path.is_file() or path.stat().st_size != record['bytes']: raise RuntimeError(f'Missing/truncated delta file: {path}')
WORK_DIR.mkdir(parents=True, exist_ok=True)
if RUNTIME_DIR.exists(): shutil.rmtree(RUNTIME_DIR)
RUNTIME_DIR.mkdir(parents=True)
run(sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--ignore-installed', '--target', RUNTIME_DIR, '--find-links', DELTA_BUNDLE / 'wheels', '-r', DELTA_BUNDLE / 'requirements-offline.txt')
project_wheels = sorted((DELTA_BUNDLE / 'wheels').glob('uit_legalir-*.whl'))
if len(project_wheels) != 1: raise RuntimeError(f'Expected one project wheel, found {project_wheels}')
run(sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--ignore-installed', '--target', RUNTIME_DIR, project_wheels[0])
runtime_env = os.environ.copy()
runtime_env['PYTHONPATH'] = str(RUNTIME_DIR)
runtime_env['PYTHONNOUSERSITE'] = '1'
print('Phase 3 delta commit:', delta_manifest['project_commit'])

In [ ]:
gpu_probe = "import torch; assert torch.cuda.is_available(), 'No CUDA GPU available'; print('GPU:', torch.cuda.get_device_name(0)); print('CUDA:', torch.version.cuda)"
run(sys.executable, '-c', gpu_probe, env=runtime_env)
contexts_source = DATASET_DIR / 'selected-contexts' / 'selected-contexts'
if not contexts_source.is_dir(): raise FileNotFoundError(f'Missing corpus: {contexts_source}')
for filename in ('train.json', TEST_FILENAME):
    if not (DATASET_DIR / filename).is_file(): raise FileNotFoundError(f'Missing input: {filename}')
def ensure_input_link(destination, source, is_directory=False):
    if destination.is_symlink():
        if destination.resolve() == source.resolve(): return
        destination.unlink()
    elif destination.exists():
        raise RuntimeError(f'Refusing to overwrite existing work file: {destination}')
    destination.symlink_to(source, target_is_directory=is_directory)
ensure_input_link(WORK_DIR / 'selected-contexts', contexts_source, is_directory=True)
for filename in ('train.json', TEST_FILENAME): ensure_input_link(WORK_DIR / filename, DATASET_DIR / filename)
test_bytes = (WORK_DIR / TEST_FILENAME).read_bytes()
test_question_count = len(json.loads(test_bytes))
test_fingerprint = __import__('hashlib').sha256(test_bytes).hexdigest()
print(f"Contexts: {sum(1 for _ in contexts_source.glob('context_*.json'))}; {TEST_LABEL} questions from {TEST_FILENAME}: {test_question_count}")

In [ ]:
import yaml
config = yaml.safe_load((DELTA_BUNDLE / 'configs' / 'kaggle_rtx_pro_6000.yaml').read_text(encoding='utf-8'))
config['paths']['public_file'] = TEST_FILENAME  # the pipeline calls this split 'public' internally
for name in ('vietlegal_harrier', 'vietnamese_embedding', 'nemotron'): config['models'][name]['local_path'] = str(PHASE2_BUNDLE / 'models' / name)
for name in ('legal_reranker', 'qwen3_reranker', 'prism_reranker'): config['models'][name]['local_path'] = str(DELTA_BUNDLE / 'models' / name)
for spec in config['models'].values(): spec['local_files_only'] = True
artifacts = WORK_DIR / config['paths']['artifacts_dir']
artifacts.mkdir(parents=True, exist_ok=True)
def seed_checkpoint(source_dir):
    if source_dir is None: return
    source_dir = Path(source_dir).resolve()
    if not source_dir.is_dir(): raise FileNotFoundError(f'Missing checkpoint artifacts: {source_dir}')
    prepare_checkpoint = source_dir / 'prepare_manifest.json'
    if not prepare_checkpoint.is_file(): raise RuntimeError(f'Checkpoint has no prepare manifest: {prepare_checkpoint}')
    prepare_metadata = json.loads(prepare_checkpoint.read_text(encoding='utf-8'))
    expected_chunking = __import__('hashlib').sha256(json.dumps(config['chunking'], ensure_ascii=False, sort_keys=True).encode('utf-8')).hexdigest()
    if prepare_metadata.get('chunking_fingerprint') != expected_chunking: raise RuntimeError('Checkpoint chunking config does not match this run')
    checkpoint_names = set()
    checkpoint_manifest = source_dir / 'model_manifest.json'
    if checkpoint_manifest.is_file(): checkpoint_names = {row['name'] for row in json.loads(checkpoint_manifest.read_text(encoding='utf-8'))['models']}
    dense_names = {'vietlegal_harrier', 'vietnamese_embedding', 'nemotron'}
    if checkpoint_names and not dense_names <= checkpoint_names: raise RuntimeError(f'Checkpoint lacks required dense retrievers: {checkpoint_names}')
    for name in dense_names:
        for required in ('vectors.npy', 'index.faiss', 'chunks.json'):
            if not (source_dir / 'dense' / name / required).is_file(): raise RuntimeError(f'Incomplete dense checkpoint: {name}/{required}')
        for required in ('vectors.npy', 'questions.json'):
            if not (source_dir / 'question_memory' / name / required).is_file(): raise RuntimeError(f'Incomplete question-memory checkpoint: {name}/{required}')
    for relative in ('corpus.jsonl', 'chunks_short.jsonl', 'chunks_long.jsonl', 'lexical_short.pkl', 'dense', 'question_memory'):
        source = source_dir / relative
        destination = artifacts / relative
        if source.exists() and not destination.exists(): destination.symlink_to(source, target_is_directory=source.is_dir())
    phase3_names = set(config['models'])
    same_phase = checkpoint_names == phase3_names
    patterns = ['prepare_manifest.json', 'train_questions.jsonl']
    if same_phase: patterns += ['first_stage_weights.json', 'final_weights*.json', 'retrieval_train.json', 'fused_train.json', 'rerank_train*.json']
    for pattern in patterns:
        for source in source_dir.glob(pattern):
            destination = artifacts / source.name
            if not destination.exists(): shutil.copy2(source, destination)
    print('Seeded', 'Phase 3' if same_phase else 'retrieval-only', 'checkpoint from:', source_dir)
seed_checkpoint(CHECKPOINT_ARTIFACTS_DIR)
input_state_path = WORK_DIR / 'inference_input_state.json'
config_fingerprint = __import__('hashlib').sha256(yaml.safe_dump(config, allow_unicode=True, sort_keys=True).encode('utf-8')).hexdigest()
input_state = {'filename': TEST_FILENAME, 'sha256': test_fingerprint, 'questions': test_question_count, 'config_sha256': config_fingerprint, 'project_commit': delta_manifest['project_commit']}
previous_input_state = json.loads(input_state_path.read_text(encoding='utf-8')) if input_state_path.is_file() else None
if previous_input_state != input_state:
    for pattern in ('public_questions.jsonl', 'retrieval_public.json', 'fused_public.json', 'rerank_public*.json'):
        for stale in artifacts.glob(pattern):
            if stale.is_file() or stale.is_symlink(): stale.unlink()
    print('Invalidated test-dependent caches:', previous_input_state, '->', input_state)
input_state_path.write_text(json.dumps(input_state, ensure_ascii=False, indent=2), encoding='utf-8')
config_path = WORK_DIR / 'kaggle_rtx_pro_6000_phase3.yaml'
config_path.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False), encoding='utf-8')
print(config_path.read_text(encoding='utf-8'))

In [ ]:
preflight = WORK_DIR / 'phase3_preflight.py'
preflight.write_text('''
import sys
from pathlib import Path
import torch
import yaml
from legalir.embeddings import load_encoder
from legalir.rerank import PairwiseReranker, CausalYesNoReranker
config = yaml.safe_load(Path(sys.argv[1]).read_text(encoding='utf-8'))
for name, spec in config['models'].items():
    if spec['role'] == 'dense':
        model = load_encoder(spec, config['runtime'])
        assert len(model.encode([spec['prompt_query'] + 'điều kiện cấp giấy phép'], convert_to_numpy=True)) == 1
        del model
        torch.cuda.empty_cache()
for name, spec in config['models'].items():
    if spec['role'] == 'pairwise_reranker': engine = PairwiseReranker(config, name)
    elif spec['role'] == 'causal_reranker': engine = CausalYesNoReranker(config, name)
    else: continue
    assert len(engine.rank('câu hỏi pháp luật', ['văn bản pháp luật liên quan', 'văn bản không liên quan'])) == 2
    engine.close()
print('Phase 3 all local model preflight tests passed.')
'''.lstrip(), encoding='utf-8')
run(sys.executable, preflight, config_path, cwd=WORK_DIR, env=runtime_env)

In [ ]:
artifacts.mkdir(parents=True, exist_ok=True)
first_stage = {'weights': {'bm25': 0.0, 'accent_char': 0.5, 'vietlegal_harrier': 2.0, 'vietnamese_embedding': 1.0, 'nemotron': 2.0, 'query_memory': 1.0, 'query_exact': 4.0}, 'rrf_k': 20}
(artifacts / 'first_stage_weights.json').write_text(json.dumps(first_stage, ensure_ascii=False, indent=2), encoding='utf-8')
base = [sys.executable, '-m', 'legalir']
def legalir(*args): run(*base, *args, cwd=WORK_DIR, env=runtime_env)
legalir('prepare', '--config', config_path, '--resume')
legalir('audit', '--config', config_path)
legalir('index', '--config', config_path, '--lexical-only', '--resume')
for model in ('vietlegal_harrier', 'vietnamese_embedding', 'nemotron'): legalir('index', '--config', config_path, '--model', model, '--resume')
legalir('tune', '--config', config_path, '--final', '--fold', '0', '--resume')
shutil.copy2(artifacts / 'final_weights_fold0.json', artifacts / 'final_weights.json')
legalir('retrieve', '--config', config_path, '--split', 'public', '--resume')
for engine in ('legal_reranker', 'qwen3_reranker', 'prism_reranker'): legalir('rerank', '--config', config_path, '--split', 'public', '--engine', engine, '--resume')
legalir('rerank', '--config', config_path, '--split', 'public', '--resume')
submission_json = WORK_DIR / f'submission_phase3_{TEST_LABEL}_tuned.json'
submission_zip = WORK_DIR / f'submission_phase3_{TEST_LABEL}_tuned.zip'
legalir('predict', '--config', config_path, '--output', submission_json, '--resume')
run('zip', '-j', submission_zip, submission_json)
print('Submission:', submission_zip)

In [ ]:
manifest = json.loads((artifacts / 'model_manifest.json').read_text(encoding='utf-8'))
final_stage = json.loads((artifacts / 'final_weights.json').read_text(encoding='utf-8'))
report = {'experiment_id': EXPERIMENT_ID, 'test_file': TEST_FILENAME, 'test_questions': test_question_count, 'test_sha256': test_fingerprint, 'evaluation': 'inference_only; labels and Recall not evaluated', 'project_commit': delta_manifest['project_commit'], 'models': manifest['models'], 'total_parameters': manifest['total_parameters'], 'first_stage': first_stage, 'final_stage': final_stage, 'submission': str(submission_zip)}
(WORK_DIR / 'phase3_report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(report, ensure_ascii=False, indent=2))